# Inference

Previously, we scored the core variant set but we did not demonstrate the workflow including the prior imputation step.
This notebook includes a full workflow for scoring new variants with FuncVEP and ClinVEP models.

---
## Impute the missing inputs using the previously trained regressors

This imputation step is necessary because the FuncVEP and ClinVEP models were trained on feature matrices with these columns already imputed, and they are not designed to handle missing values in these inputs. Leaving NaN values in these features at prediction time could lead to unstable or biased model behaviour.

To prevent cascading imputation, each model uses a frozen copy of the original (pre-imputation) feature matrix as predictors, so imputed values are never used as inputs for subsequent imputations.

In [ ]:
import os
import joblib
import pandas as pd

def impute_missing_values(df):
    imputer_dir = "../models/imputation"

    with open("../resources/feature_lists/columns_to_impute.txt", "r") as f:
        columns_to_impute = [line.strip() for line in f.readlines()]

    with open("../resources/feature_lists/veps_excluded_due_to_unavailable_training_sets.txt", "r") as f:
        no_training_set = [line.strip() for line in f]

    df = df.copy()
    df.columns = df.columns.str.replace(" ", "_")

    id_cols = {"ID", "ensg", "gene", "enst"}
    present_id_cols = list(id_cols & set(df.columns))
    if present_id_cols:
        df[present_id_cols] = df[present_id_cols].astype(str)

    numeric_cols = [c for c in df.columns if c not in id_cols]
    if numeric_cols:
        df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

    # Frozen view of original numeric inputs so we never use imputed values as predictors
    predictors_df = df.copy()

    for col in columns_to_impute:
        if col not in df.columns:
            print(f"Skipping {col}: not found in dataframe.")
            continue

        if col in no_training_set:
            print(f"Skipping {col}: excluded from imputation.")
            continue

        model_path = os.path.join(imputer_dir, f"{col}_imputer.pkl")
        if not os.path.exists(model_path):
            print(f"Skipping {col}: imputation model not found.")
            continue

        missing_mask = df[col].isna()
        n_missing = int(missing_mask.sum())
        if n_missing == 0:
            continue

        imputer = joblib.load(model_path)
        predictors = list(imputer.feature_name_)

        # We require all predictors that the model was trained on
        missing_predictors = set(predictors) - set(predictors_df.columns)
        if missing_predictors:
            raise ValueError(
                f"Imputation model for {col} expects missing predictors: "
                f"{sorted(missing_predictors)}"
            )

        X_pred = predictors_df.loc[missing_mask, predictors]

        if X_pred.empty:
            print(f"Skipping {col}: no rows with usable predictors.")
            continue

        preds = imputer.predict(X_pred)
        df.loc[missing_mask, col] = preds
        print(f"Imputed {n_missing} missing values in {col}.")

    return df


## Run FuncVEP and ClinVEP inference

For each model, we load the trained LightGBM classifier and apply it only to variants that were not used in its training set.

We also enforce exact feature consistency with the training configuration. 


In [2]:
import os
import glob
import joblib
import pandas as pd

def run_inference(model_name, df, dataset_name=None):
    model_dir = f"../models/{model_name}"

    df = df.copy()
    df.columns = df.columns.str.replace(" ", "_")

    id_column = "ID"
    ensg_column = "ensg"

    df[id_column] = df[id_column].astype(str)
    if ensg_column in df.columns:
        df[ensg_column] = df[ensg_column].astype(str)

    trained_on = pd.read_csv(os.path.join(model_dir, "training_set.txt"), sep="\t")
    trained_on[id_column] = trained_on[id_column].astype(str)

    if ensg_column in trained_on.columns and ensg_column in df.columns:
        trained_on[ensg_column] = trained_on[ensg_column].astype(str)
        train_idx = pd.MultiIndex.from_frame(trained_on[[id_column, ensg_column]])
        test_idx = pd.MultiIndex.from_frame(df[[id_column, ensg_column]])
        keep_mask = ~test_idx.isin(train_idx)
        df = df.loc[keep_mask].copy()
    else:
        df = df[~df[id_column].isin(trained_on[id_column])].copy()

    lgb_model = joblib.load(os.path.join(model_dir, "model.pkl"))

    feature_path = os.path.join(model_dir, "training_features.txt")
    if os.path.exists(feature_path):
        trained_features = pd.read_csv(feature_path, sep="\t")["Feature"].tolist()
    else:
        trained_features = list(lgb_model.feature_name_)

    missing_features = set(trained_features) - set(df.columns)
    if missing_features:
        raise ValueError(
            f"Missing features in inference input for {model_name}: "
            f"{sorted(missing_features)}"
        )

    df[trained_features] = df[trained_features].apply(pd.to_numeric, errors="coerce")
    X = df[trained_features]

    df[model_name] = lgb_model.predict_proba(X)[:, 1]
    return df[[id_column, ensg_column, model_name]]


Finally, we call a wrapper function that imputes inputs, runs all six FuncVEP and ClinVEP models, and saves the scores for downstream analyses.

In [3]:
import os
import pandas as pd

def run_all_models(df, output_filename, output_dir="../results/predictions"):
    df_input = df.copy()
    df_input.columns = df_input.columns.str.replace(" ", "_")

    df_imputed = impute_missing_values(df_input)

    model_names = ["FuncVEP_CTI", "FuncVEP_CTE", "FuncVEP_SP", "ClinVEP_CTI", "ClinVEP_CTE", "ClinVEP_SP"]
    preds = {}
    for model_name in model_names:
        preds[model_name] = run_inference(model_name, df_imputed)

    combined = preds[model_names[0]]
    for model_name in model_names[1:]:
        combined = combined.merge(
            preds[model_name],
            on=["ID", "ensg"],
            how="outer",
        )

    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, output_filename)
    combined.to_csv(output_path, sep="\t", index=False)
    print(f"Saved combined predictions to {output_path}")

    return combined


In [4]:
proteingym_df = pd.read_parquet("../data/intermediate/proteingym_feature_matrix.parquet")

combined_pg = run_all_models(
    proteingym_df,
    output_filename="proteingym_scores.txt",
    output_dir="../results/predictions",
)


Imputed 69028 missing values in glm_CaddDeogenRevel.
Imputed 9143 missing values in glm_AlphDeogenRevel.
Imputed 9143 missing values in glm_AlphCaddDeogen.
Imputed 9133 missing values in glm_AlphRevelCadd.
Imputed 8188 missing values in glm_AlphRevel.
Imputed 6074 missing values in MutPred_score.
Imputed 7465 missing values in glm_DeogenRevel.
Imputed 6518 missing values in glm_RevelCadd.
Imputed 9394 missing values in REVEL_score.
Imputed 9541 missing values in MetaRNN_score.
Imputed 3566 missing values in M_CAP_score.
Imputed 23382 missing values in EVH_epistatic.
Imputed 23382 missing values in EVH_independent.
Imputed 18838 missing values in VARITY_ER_LOO_score.
Imputed 18838 missing values in VARITY_R_LOO_score.
Imputed 18838 missing values in VARITY_ER_score.
Imputed 18838 missing values in VARITY_R_score.
Imputed 2783 missing values in MutFormer_score.
Imputed 3161 missing values in MetaLR_score.
Imputed 3161 missing values in MetaSVM_score.
Imputed 6899 missing values in glm_Al

/tmp/ipykernel_474952/3034851203.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[model_name] = lgb_model.predict_proba(X)[:, 1]
/tmp/ipykernel_474952/3034851203.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[model_name] = lgb_model.predict_proba(X)[:, 1]
/tmp/ipykernel_474952/3034851203.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To ge

Saved combined predictions to ../results/predictions/proteingym_scores.txt
